# MACHINE LEARNING — LAB 6
## Logistic Regression & KNN Classification

### Telco Customer Churn Prediction

**Student Name:** Alia Kalwar  
**Student ID:** ____________________  
**Section:** ____________________  
**GitHub Profile:** ____________________  
**Kaggle Profile:** ____________________  

**Dataset:** Telco Customer Churn  
**Target Variable:** Churn (Yes/No)  
**Tools:** Python, Pandas, NumPy, Scikit-learn, Matplotlib, Google Colab

---

## Learning Objectives

By the end of this lab, we will:

1. Verify the cleaned Telco Customer Churn dataset.
2. Prepare numerical and categorical features.
3. Apply feature scaling correctly without data leakage.
4. Train and evaluate Logistic Regression.
5. Train KNN using several values of K.
6. Select an appropriate K based on performance.
7. Compare Decision Tree, Random Forest, Logistic Regression, and KNN.
8. Interpret Logistic Regression coefficients using odds ratios.
9. Select a final model using evidence from the comparison.
10. Write a final conclusion and discuss limitations.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

# 1. Problem Definition & Dataset Verification

The problem is to predict whether a Telco customer will churn or stay with the company.

The target variable is `Churn`:

- `Yes` = customer churned
- `No` = customer did not churn

The cleaned Telco Customer Churn dataset from the previous labs is used.

In [ ]:
# Load the cleaned dataset

file_path = "/content/clean_churn.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

### Dataset Verification

We verify the number of rows and columns, data types, missing values, and the target distribution before preparing the data.

In [ ]:
# Verify shape, columns, data types, and missing values

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Check target variable distribution

print("Churn value counts:")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

### Interpretation

The target variable contains two classes, `Yes` and `No`. The number of non-churn customers is larger than the number of churn customers, so the dataset is imbalanced.

Because of this class imbalance, accuracy alone should not be used to judge the models. Precision, recall, F1-score, and the confusion matrix will also be considered.

### Lab 3 and Lab 4 Benchmarks

The previous labs used Decision Tree and Random Forest for the same Telco Churn problem.

For a fair comparison in this notebook, Decision Tree and Random Forest are retrained using the same train/test split used for today's Logistic Regression and KNN models.

If your instructor requires the exact numerical results from your submitted Lab 3/4 notebooks, those previous values can also be reported separately.

# 2. Feature Scaling & Preparation

Categorical variables must be converted into numerical form before they can be used by the classifiers.

One-hot encoding is used for categorical features.

Feature scaling is especially important for KNN because KNN uses distance between observations. Features with larger numerical ranges can dominate the distance calculation.

Scaling also helps Logistic Regression by putting numerical features on comparable scales.

Decision Trees and Random Forests do not require feature scaling because they make decisions using feature thresholds rather than distance calculations.

The scaler is fitted only on the training data to prevent data leakage.

In [ ]:
# Separate features and target

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

print("Original feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget encoding:")
print(y.value_counts())

In [ ]:
# Drop identifier columns if present

identifier_columns = ["customerID", "CustomerID", "customer_id"]

columns_to_drop = [
    column for column in identifier_columns
    if column in X.columns
]

if columns_to_drop:
    X = X.drop(columns=columns_to_drop)
    print("Dropped identifier columns:", columns_to_drop)
else:
    print("No identifier column found.")

In [ ]:
# One-hot encode categorical variables

X_encoded = pd.get_dummies(
    X,
    drop_first=True
)

# Convert boolean columns to integers if necessary
bool_columns = X_encoded.select_dtypes(include=["bool"]).columns

if len(bool_columns) > 0:
    X_encoded[bool_columns] = X_encoded[bool_columns].astype(int)

print("Shape after encoding:", X_encoded.shape)
print("Number of final features:", X_encoded.shape[1])

display(X_encoded.head())

In [ ]:
# Train/test split
# Stratification keeps the churn proportion approximately similar in both sets.

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set :", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

In [ ]:
# StandardScaler
# IMPORTANT: fit only on training data

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape :", X_test_scaled.shape)

### Why is the scaler fitted only on the training data?

The scaler is fitted only on the training data because the test data must remain unseen during model preparation.

If the scaler were fitted on the complete dataset, information from the test set would be used to calculate the scaling parameters. This would cause data leakage and could produce an unrealistically optimistic evaluation.

Therefore:

- `fit_transform()` is used on training data.
- `transform()` is used on testing data.

# 3. Logistic Regression

Logistic Regression is trained on the scaled training data.

The model predicts the probability of belonging to the churn class.

We evaluate the model using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- Predicted probabilities using `predict_proba()`

In [ ]:
# Train Logistic Regression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression trained successfully.")

In [ ]:
# Logistic Regression predictions

y_pred_lr = logistic_model.predict(X_test_scaled)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr, zero_division=0)
lr_recall = recall_score(y_test, y_pred_lr, zero_division=0)
lr_f1 = f1_score(y_test, y_pred_lr, zero_division=0)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1-score : {lr_f1:.4f}")

In [ ]:
# Logistic Regression confusion matrix

lr_cm = confusion_matrix(y_test, y_pred_lr)

print("Confusion Matrix:")
print(lr_cm)

ConfusionMatrixDisplay(
    confusion_matrix=lr_cm,
    display_labels=["No Churn", "Churn"]
).plot()

plt.title("Logistic Regression Confusion Matrix")
plt.show()

### Logistic Regression Interpretation

Accuracy represents the overall proportion of correct predictions.

Precision tells us how many customers predicted as churn actually churned.

Recall tells us how many of the actual churn customers were correctly identified.

F1-score combines precision and recall into one measure and is useful for imbalanced classification.

The confusion matrix provides the counts of true negatives, false positives, false negatives, and true positives.

In [ ]:
# Look at predicted probabilities for a few test customers

probabilities = logistic_model.predict_proba(X_test_scaled)

probability_table = pd.DataFrame(
    probabilities[:10],
    columns=["Probability_No_Churn", "Probability_Churn"]
)

probability_table["Predicted_Class"] = y_pred_lr[:10]

display(probability_table)

### Understanding `predict_proba()`

`predict_proba()` gives the estimated probability of each class.

For example, a churn probability of 0.80 means that the model estimates an 80% probability of churn for that customer.

`predict()` converts these probabilities into the final class prediction using the classifier's decision threshold.

# 4. KNN & Choosing K

K-Nearest Neighbors classifies an observation according to the classes of nearby training observations.

The value of K controls model complexity:

- Very small K can be sensitive to noise and may overfit.
- Very large K can average over too many observations and may underfit.

The following K values will be tested:

`1, 3, 5, 7, 9, 11, 15, 21`

In [ ]:
# Test multiple K values

k_values = [1, 3, 5, 7, 9, 11, 15, 21]

knn_results = []

for k in k_values:

    knn_model = KNeighborsClassifier(
        n_neighbors=k
    )

    knn_model.fit(
        X_train_scaled,
        y_train
    )

    y_pred_knn = knn_model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred_knn)
    precision = precision_score(y_test, y_pred_knn, zero_division=0)
    recall = recall_score(y_test, y_pred_knn, zero_division=0)
    f1 = f1_score(y_test, y_pred_knn, zero_division=0)

    knn_results.append({
        "K": k,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

knn_results_df = pd.DataFrame(knn_results)

display(
    knn_results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)

In [ ]:
# Plot Accuracy against K

plt.figure(figsize=(8, 5))

plt.plot(
    knn_results_df["K"],
    knn_results_df["Accuracy"],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy for Different Values of K")
plt.grid(True)
plt.show()

In [ ]:
# Plot F1-score against K

plt.figure(figsize=(8, 5))

plt.plot(
    knn_results_df["K"],
    knn_results_df["F1-Score"],
    marker="o"
)

plt.xlabel("K")
plt.ylabel("F1-Score")
plt.title("KNN F1-Score for Different Values of K")
plt.grid(True)
plt.show()

In [ ]:
# Select K with the highest F1-score

best_knn_row = knn_results_df.loc[
    knn_results_df["F1-Score"].idxmax()
]

best_k = int(best_knn_row["K"])

print("Selected K:", best_k)
print(f"Accuracy : {best_knn_row['Accuracy']:.4f}")
print(f"Precision: {best_knn_row['Precision']:.4f}")
print(f"Recall   : {best_knn_row['Recall']:.4f}")
print(f"F1-score : {best_knn_row['F1-Score']:.4f}")

### K Selection Interpretation

K is selected using the highest F1-score among the tested values.

The smallest K is not automatically selected because a very small K can be sensitive to individual noisy observations.

The largest K is not automatically selected because a very large K can make the model too general.

The selected K gives the strongest balance between precision and recall among the tested values.

In [ ]:
# Train final KNN using the selected K

final_knn = KNeighborsClassifier(
    n_neighbors=best_k
)

final_knn.fit(
    X_train_scaled,
    y_train
)

y_pred_knn = final_knn.predict(X_test_scaled)

knn_accuracy = accuracy_score(y_test, y_pred_knn)
knn_precision = precision_score(y_test, y_pred_knn, zero_division=0)
knn_recall = recall_score(y_test, y_pred_knn, zero_division=0)
knn_f1 = f1_score(y_test, y_pred_knn, zero_division=0)

print("Final KNN Results")
print("-----------------")
print(f"Selected K: {best_k}")
print(f"Accuracy : {knn_accuracy:.4f}")
print(f"Precision: {knn_precision:.4f}")
print(f"Recall   : {knn_recall:.4f}")
print(f"F1-score : {knn_f1:.4f}")

In [ ]:
# Final KNN confusion matrix

knn_cm = confusion_matrix(y_test, y_pred_knn)

print("KNN Confusion Matrix:")
print(knn_cm)

ConfusionMatrixDisplay(
    confusion_matrix=knn_cm,
    display_labels=["No Churn", "Churn"]
).plot()

plt.title(f"KNN Confusion Matrix (K={best_k})")
plt.show()

### Final KNN Interpretation

The final KNN model is evaluated using accuracy, precision, recall, and F1-score.

The confusion matrix shows how many customers were correctly and incorrectly classified into the churn and no-churn classes.

The selected K provides a balance between overfitting and underfitting compared with the other tested K values.

# 5. Model Comparison (4 Models)

The following four classifiers are compared using the same train/test split:

1. Decision Tree
2. Random Forest
3. Logistic Regression
4. KNN

All four models are evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

In [ ]:
# Train Decision Tree

decision_tree = DecisionTreeClassifier(
    random_state=42
)

decision_tree.fit(
    X_train,
    y_train
)

y_pred_dt = decision_tree.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt, zero_division=0)
dt_recall = recall_score(y_test, y_pred_dt, zero_division=0)
dt_f1 = f1_score(y_test, y_pred_dt, zero_division=0)

print("Decision Tree trained successfully.")

In [ ]:
# Train Random Forest

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(
    X_train,
    y_train
)

y_pred_rf = random_forest.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, zero_division=0)
rf_recall = recall_score(y_test, y_pred_rf, zero_division=0)
rf_f1 = f1_score(y_test, y_pred_rf, zero_division=0)

print("Random Forest trained successfully.")

In [ ]:
# Create the four-model comparison table

comparison_df = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "Logistic Regression",
        f"KNN (K={best_k})"
    ],
    "Accuracy": [
        dt_accuracy,
        rf_accuracy,
        lr_accuracy,
        knn_accuracy
    ],
    "Precision": [
        dt_precision,
        rf_precision,
        lr_precision,
        knn_precision
    ],
    "Recall": [
        dt_recall,
        rf_recall,
        lr_recall,
        knn_recall
    ],
    "F1-Score": [
        dt_f1,
        rf_f1,
        lr_f1,
        knn_f1
    ]
})

display(
    comparison_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)

In [ ]:
# Identify the model with the highest value for each metric

best_accuracy_model = comparison_df.loc[
    comparison_df["Accuracy"].idxmax(), "Model"
]

best_precision_model = comparison_df.loc[
    comparison_df["Precision"].idxmax(), "Model"
]

best_recall_model = comparison_df.loc[
    comparison_df["Recall"].idxmax(), "Model"
]

best_f1_model = comparison_df.loc[
    comparison_df["F1-Score"].idxmax(), "Model"
]

print("Best Accuracy Model :", best_accuracy_model)
print("Best Precision Model:", best_precision_model)
print("Best Recall Model   :", best_recall_model)
print("Best F1 Model       :", best_f1_model)

### Model Comparison Interpretation

The four models are compared using all four required metrics.

Accuracy measures overall correctness.

Precision measures how reliable positive churn predictions are.

Recall measures how many actual churn customers were identified.

F1-score balances precision and recall.

Because the dataset is imbalanced, accuracy should not be used alone. Recall and F1-score are especially useful when the goal is to identify customers who may churn.

### Which Metric Should Carry More Weight?

The Telco Churn dataset contains more non-churn customers than churn customers.

If the business objective is to identify customers who may leave, recall is important because a false negative means that an actual churn customer was not identified.

F1-score is also useful because it balances precision and recall.

Therefore, the final model decision should consider F1-score and recall together with the other metrics instead of relying only on accuracy.

# 6. Coefficient Interpretation

Logistic Regression coefficients describe the effect of features on the log-odds of churn.

- Positive coefficient → higher log-odds of churn.
- Negative coefficient → lower log-odds of churn.

The coefficient can be converted to an odds ratio using:

`Odds Ratio = exp(coefficient)`

Interpretation:

- Odds Ratio > 1 → higher odds of churn.
- Odds Ratio < 1 → lower odds of churn.
- Odds Ratio = 1 → no change in odds.

For one-hot encoded categorical variables, the effect is relative to the reference category that was dropped.

In [ ]:
# Extract Logistic Regression coefficients

coefficient_table = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": logistic_model.coef_[0]
})

coefficient_table["Odds Ratio"] = np.exp(
    coefficient_table["Coefficient"]
)

coefficient_table["Absolute Coefficient"] = (
    coefficient_table["Coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "Absolute Coefficient",
    ascending=False
)

display(
    coefficient_table[
        ["Feature", "Coefficient", "Odds Ratio"]
    ].head(15).style.format({
        "Coefficient": "{:.4f}",
        "Odds Ratio": "{:.4f}"
    })
)

In [ ]:
# Display the four strongest features

top_four = coefficient_table.head(4)

display(
    top_four[
        ["Feature", "Coefficient", "Odds Ratio"]
    ].style.format({
        "Coefficient": "{:.4f}",
        "Odds Ratio": "{:.4f}"
    })
)

In [ ]:
# Automatically generate interpretations for the four strongest features

print("Interpretation of the four strongest Logistic Regression features:")
print()

for _, row in top_four.iterrows():

    feature = row["Feature"]
    coefficient = row["Coefficient"]
    odds_ratio = row["Odds Ratio"]

    if coefficient > 0:
        print(
            f"{feature}: coefficient = {coefficient:.4f}, "
            f"odds ratio = {odds_ratio:.4f}. "
            f"The feature is associated with higher odds of churn, "
            f"holding the other features constant."
        )
    else:
        print(
            f"{feature}: coefficient = {coefficient:.4f}, "
            f"odds ratio = {odds_ratio:.4f}. "
            f"The feature is associated with lower odds of churn, "
            f"holding the other features constant."
        )

### Coefficient Interpretation

The coefficient table identifies features with the strongest effects in the Logistic Regression model.

A positive coefficient means that the feature is associated with higher churn odds, while a negative coefficient means that it is associated with lower churn odds.

The odds ratio makes the result easier to interpret because it expresses the effect multiplicatively.

For categorical one-hot encoded features, the comparison is against the reference category that was omitted during encoding.

### Comparison with Earlier Labs

The Logistic Regression coefficients can be compared with the patterns found during EDA and Decision Tree feature importance analysis.

Features that appear important across several methods provide stronger evidence that they are useful predictors of churn.

Some differences are expected because Logistic Regression models linear relationships in log-odds, while Decision Trees can capture non-linear patterns and interactions.

# 7. Final Model Selection & Conclusion

In [ ]:
# Select the model with the highest F1-score

final_model_row = comparison_df.loc[
    comparison_df["F1-Score"].idxmax()
]

final_model_name = final_model_row["Model"]

print("Model with the highest F1-score:")
print(final_model_name)

print("\nPerformance:")
print(f"Accuracy : {final_model_row['Accuracy']:.4f}")
print(f"Precision: {final_model_row['Precision']:.4f}")
print(f"Recall   : {final_model_row['Recall']:.4f}")
print(f"F1-score : {final_model_row['F1-Score']:.4f}")

### Final Model Selection

The final model is selected from the four candidates using the evidence in the comparison table.

The selection is not based on accuracy alone because the churn classes are imbalanced.

F1-score and recall receive strong attention because identifying actual churn customers is important in this problem.

The model with the strongest overall balance of the required metrics is selected as the final candidate for this experiment.

In [ ]:
# Display the final comparison table

display(
    comparison_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)

## Final Conclusion

This lab compared four classification algorithms for predicting customer churn: Decision Tree, Random Forest, Logistic Regression, and KNN.

Feature scaling was especially important for KNN because KNN depends on distances between observations, while Logistic Regression also benefits from numerical features being on comparable scales.

Logistic Regression provided interpretable coefficients that could be converted into odds ratios to understand how different features were associated with churn.

KNN performance changed as K changed, showing why an appropriate K should be selected instead of automatically choosing the smallest or largest value.

The four models were compared using accuracy, precision, recall, and F1-score rather than accuracy alone.

The final model was selected from the comparison results, with particular attention to F1-score and recall because the target classes are imbalanced.

Possible future improvements include cross-validation, hyperparameter tuning, class-imbalance techniques, feature selection, and additional preprocessing.

# Final Lab Checklist

Before submitting, make sure:

- [x] Student information block is present.
- [x] Dataset is loaded and verified.
- [x] Target variable is converted to 0/1.
- [x] Identifier columns are removed.
- [x] Categorical variables are one-hot encoded.
- [x] Train/test split is performed.
- [x] StandardScaler is fitted only on training data.
- [x] Logistic Regression is trained.
- [x] Logistic Regression has accuracy, precision, recall, F1-score.
- [x] Logistic Regression confusion matrix is included.
- [x] `predict_proba()` is demonstrated.
- [x] Multiple K values are tested.
- [x] KNN results table is included.
- [x] KNN accuracy and F1 plots are included.
- [x] Best K is selected.
- [x] Final KNN metrics and confusion matrix are included.
- [x] Decision Tree is included.
- [x] Random Forest is included.
- [x] Four-model comparison table is included.
- [x] Logistic Regression coefficients are extracted.
- [x] Odds ratios are calculated.
- [x] At least four features are interpreted.
- [x] Final model selection is explained.
- [x] Final conclusion is included.